# Sentiment Analysis using ChatGPT 5

In [ ]:
import openai
from openai import OpenAI
import json
import os
from dotenv import load_dotenv

# define the environment path. it is at the parent folder
env_path ="../.env"

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

MODEL = "gpt-5-chat-latest"

def classify_sentiment(review_text: str) -> dict:
    system = {
        "role": "system",
        "content": (
            "You are a precise sentiment classifier. "
            "Return ONLY valid JSON with keys: label, confidence, rationale. "
            "label ∈ {Positive, Negative, Neutral}. "
            "confidence is a float in [0,1]. rationale ≤ 20 words."
        ),
    }
    user = {
        "role": "user",
        "content": [
            {"type": "text",
             "text": (
                 "Classify the sentiment of the following review. "
                 "Return JSON: {\"label\":\"Positive|Negative|Neutral\",\"confidence\":0-1,\"rationale\":\"...\"}.\n\n"
                 f"Review:\n{review_text}"
             )},
        ],
    }

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[system, user],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

# Example

print(classify_sentiment("Service was fast and the food was amazing! Will come back."))

# We will apply the above functions to the boston airbnb reviews we used in week 4

In [ ]:
import pandas as pd

df=pd.read_csv('data/boston_airbnb_reviews_with_human_labels.csv')

df.head()

In [ ]:
df.shape

In [ ]:
df['ai_output']=df['comments'].apply(classify_sentiment)

In [ ]:
df.head()

In [ ]:
# Extract sentiment from ai_label

import re

df["ai_label"] = df["ai_output"].astype(str).str.extract(r"'label':\s*'([^']*)'")

df.head()

## Compare AI label with human label

This is a higher agreement/accuracy between GPT 5 and my evaluation. We only have one disagreement.
GPT labels one of my nerual reviews as positive.

In [ ]:
import os
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# create a function to generate confusion matrix and classification report for sentiment analysis

def classification_evaluation(y_true, y_pred):
    classes = ["Positive", "Neutral", "Negative"]

    # generate Confusion matrix (fixed label order)
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap="Blues", values_format="d")
    plt.title("Confusion Matrix - Sentiment Analysis")
    plt.show()

    # generate classification report with the SAME labels & names. Use three decimal places
    print(classification_report(y_true, y_pred, labels=classes, target_names=classes, digits=3))

In [ ]:
# Evaluate the model performance

y_true = df['human_label']
y_pred = df['ai_label']
    
classification_evaluation(y_true, y_pred)

# Practice On your Own

Apply GPT-5 to evaluate sentiment on the dataset where you manually labeled sentiment in Week 4. Compare its performance with the BERT models from Week 4 and discuss whether the predictions show any improvement.